# Clef decision training with `DecisionTrainer` and Accelerate

This notebook trains a Clef decision model with TRL's experimental `DecisionTrainer`, on one GPU or on several with
Accelerate, and then lets you use it. In order:

1. install the two forks, pinned;
2. write one configuration and train with cross-entropy (`accelerate launch`, in fresh processes);
3. continue that model with an RLCD phase (cross-entropy first, then RLCD);
4. read the metrics of both runs;
5. ask the trained model questions;
6. optionally: convert an older checkpoint, time training steps, let a judge model watch the evaluations.

Run it on a Linux machine with CUDA. With `QUICK = True` (the default) each stage is a short run on a sample, so the
whole notebook takes minutes; set `QUICK = False` for the full recipe. Training never runs in this kernel: the kernel
only touches the GPU in "Ask the trained model".

## Settings

This is the only cell to edit.

In [ ]:
from pathlib import Path

# Where the code comes from: your forks, pinned to the commits this notebook was written against
TRANSFORMERS_SOURCE = "https://github.com/ericlewis/transformers/archive/b5c4f2e4218b5f4f1a47ee7ad5d31a03a989987f.tar.gz"
TRL_SOURCE = "https://github.com/ericlewis/trl/archive/9ff7d39bea8ee6a73b15f18e7f5a60de7fb8479a.tar.gz"
FLA_SOURCE = "https://github.com/fla-org/flash-linear-attention/archive/9c102739e348968753d86bb729c57dd6efa1c413.tar.gz"
INSTALL = True  # False when this environment already has all three

# What to train
MODEL = "MinimaLabs/KLM-1.6"
MODEL_REVISION = "8fabc98abe194f7450cf19ed743caa7e85a4e5b0"
DATASET = "LocalLLaMA/typed-decisions"  # rows with `state`, `questions` and `gold`
DATASET_CONFIG = "all"
DATASET_REVISION = None
EVAL_SPLIT = None  # None: hold out 5% of the training split. Or the name of a split, such as "validation"
HEAD = "cosine"  # "cosine" (small, faster steps) or "joint" (Clef's original head)
MODE = "full"  # "full", "head" (backbone frozen) or "lora"

# How to run it
QUICK = True  # True: a short run on a sample, to see every stage work. False: the full recipe
NUM_GPUS = None  # None: every visible GPU (CUDA_VISIBLE_DEVICES is honoured)
GLOBAL_BATCH_SIZE = 32  # records per optimizer step, split evenly across the GPUs
EPOCHS = 3
LEARNING_RATE = 1e-4  # backbone
HEAD_LEARNING_RATE = 3e-4
MAX_LENGTH = 2048  # tokens per prompt; a longer record has the end of its state cut
EVAL_EVERY_STEPS = 12
ACTIVATION_CHECKPOINTING = True  # less memory, slower steps
BACKBONE_ATTENTION = "kernels-community/flash-attn2@291664546de978ab2bb730554252e5e2c1099e4f"  # on CUDA; None: default
SEED = 3407

# After cross-entropy: the RLCD phase
RLCD_PHASE = True
RLCD_EPOCHS = 4
RLCD_SIGMAS = [0.4, 0.3, 0.2, 0.1]  # noise of each RLCD epoch
RLCD_WEIGHT = 0.25  # the loss of the phase is cross-entropy + RLCD_WEIGHT x the RLCD term

# Optional extras
EXISTING_CHECKPOINT = None  # a folder in the older layout (backbone + joint_head.safetensors) to convert and try
RUN_BENCHMARK = False  # time training steps on this machine
JEV_MONITOR = None  # "typesafe", "openrouter" or a System One URL: a judge watches the evaluations (advise only)
FLA_PRESET_DIR = None  # a folder of saved FLA kernel presets to reuse for every batch size (see the notes at the end)

WORK_DIR = Path("clef_decision_run").resolve()

## Install

The forks go in last and on their own, so the torch and CUDA build of this machine is left as it is.

In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*packages, forks=False):
    """Install with pip, or with uv in an environment that has no pip."""
    if importlib.util.find_spec("pip") is not None:
        command = [sys.executable, "-m", "pip", "install", "-q", *(["--no-deps", "--force-reinstall"] if forks else [])]
    else:
        command = ["uv", "pip", "install", "--python", sys.executable, "-q", *(["--no-deps", "--reinstall"] if forks else [])]
    subprocess.run([*command, *packages], check=True)


if INSTALL:
    # The released packages first, for their dependencies
    pip_install("transformers", "trl", "accelerate>=1.15.0", "datasets", "peft", "kernels", "einops", "pyyaml")
    # Then the forks and FLA (KLM's kernels) over them
    pip_install(TRANSFORMERS_SOURCE, TRL_SOURCE, FLA_SOURCE, forks=True)

check = (
    "import accelerate, transformers, trl; from transformers.loss.loss_utils import LOSS_MAPPING; "
    "assert 'ClefForDecision' in LOSS_MAPPING, 'this transformers has no Clef training support: is the fork installed?'; "
    "from trl.experimental.decision import DecisionTrainer, RlcdScheduleCallback; "
    "print('transformers', transformers.__version__, '| trl', trl.__version__, '| accelerate', accelerate.__version__)"
)
installed = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
print(installed.stdout.strip() or installed.stderr.strip()[-2000:])
if installed.returncode != 0:
    raise RuntimeError("The pinned forks are not importable in this environment; see the message above")

## The example scripts and this machine

Training is done by the Clef example of the TRL fork, fetched here at the pinned commit. The GPUs are counted in a
separate process, so this kernel stays free of CUDA.

In [ ]:
import io
import json
import os
import tarfile
import urllib.request

WORK_DIR.mkdir(parents=True, exist_ok=True)
# Compiled kernels are kept in the run folder, for the training processes and for this kernel alike
os.environ.setdefault("TRITON_CACHE_DIR", str(WORK_DIR / "cache" / "triton"))
EXAMPLES = WORK_DIR / "trl"
TRAIN_SCRIPT = EXAMPLES / "examples/decisions/clef_typed_decisions.py"
BENCHMARK_SCRIPT = EXAMPLES / "examples/decisions/benchmark_clef.py"
if not TRAIN_SCRIPT.exists():
    with urllib.request.urlopen(TRL_SOURCE) as response:
        archive = tarfile.open(fileobj=io.BytesIO(response.read()), mode="r:gz")
    for member in archive.getmembers():
        parts = Path(member.name).parts[1:]  # without the top folder of the archive
        if member.isfile() and parts[:2] == ("examples", "decisions"):
            target = EXAMPLES.joinpath(*parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.extractfile(member).read())

probe = (
    "import json, torch; n = torch.cuda.device_count() if torch.cuda.is_available() else 0; "
    "print(json.dumps({'gpus': n, 'names': [torch.cuda.get_device_name(i) for i in range(n)], 'torch': torch.__version__}))"
)
found = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, check=True)
machine = json.loads(found.stdout.strip().splitlines()[-1])
ON_GPU = machine["gpus"] > 0
if NUM_GPUS is not None and NUM_GPUS > machine["gpus"]:
    raise ValueError(f"NUM_GPUS is {NUM_GPUS}, and this machine shows {machine['gpus']}")
PROCESSES = (NUM_GPUS or machine["gpus"]) if ON_GPU else 1
if ON_GPU and GLOBAL_BATCH_SIZE % PROCESSES:
    raise ValueError(f"GLOBAL_BATCH_SIZE {GLOBAL_BATCH_SIZE} does not divide evenly across {PROCESSES} GPUs")
PER_DEVICE_BATCH = GLOBAL_BATCH_SIZE // PROCESSES if ON_GPU else 2

print("torch", machine["torch"], "|", machine["names"] or "no CUDA GPU: a tiny CPU run, only to check the stages")
print(f"{PROCESSES} process(es) x {PER_DEVICE_BATCH} records = {PROCESSES * PER_DEVICE_BATCH} records per step")

## The configuration

One YAML file per run, with the same settings the example's own configuration uses. `write_config` is used again
for the RLCD phase.

In [ ]:
import yaml


def write_config(name, **changes):
    """Write `<name>.yaml` for a run whose output goes to `WORK_DIR/<name>`; `changes` override, `None` removes."""
    config = {
        "model_name_or_path": MODEL,
        "model_revision": MODEL_REVISION,
        "dataset_name": DATASET,
        "dataset_config": DATASET_CONFIG,
        "dataset_revision": DATASET_REVISION,
        "dataset_test_split": EVAL_SPLIT,
        "output_dir": str(WORK_DIR / name),
        "dtype": "float32",  # float32 weights under bf16 autocast, which KLM's fused kernels need
        "attn_implementation": BACKBONE_ATTENTION if ON_GPU else None,  # applied to the backbone only
        "mode": MODE,
        "head_type": HEAD,
        "max_length": MAX_LENGTH,
        "truncation": "state",
        "permute_fields": True,  # training batches reorder the questions; evaluation never does
        "per_device_train_batch_size": PER_DEVICE_BATCH,
        "per_device_eval_batch_size": 8 if ON_GPU else 2,
        "num_train_epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "head_learning_rate": HEAD_LEARNING_RATE,
        "optim": "adamw_torch_fused" if ON_GPU else "adamw_torch",
        "weight_decay": 0.05,
        "max_grad_norm": 1.0,
        "lr_scheduler_type": "cosine",
        "warmup_steps": 0.03,
        "activation_checkpointing": ACTIVATION_CHECKPOINTING and ON_GPU,
        "eval_strategy": "steps",
        "eval_steps": EVAL_EVERY_STEPS,
        "logging_steps": EVAL_EVERY_STEPS,
        "save_strategy": "best",  # a checkpoint at every evaluation that improves the metric below
        "save_total_limit": 1,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_selection_score",
        "greater_is_better": True,
        "bf16": ON_GPU,
        "use_cpu": None if ON_GPU else True,
        "dataloader_num_workers": 4 if ON_GPU else 0,
        "disable_tqdm": True,
        "seed": SEED,
        "report_to": "none",
        "jev_monitor": JEV_MONITOR,
        "jev_monitor_mode": "advise" if JEV_MONITOR else None,
    }
    if QUICK and ON_GPU:  # 20 steps per epoch, 2 epochs
        config.update(max_train_samples=GLOBAL_BATCH_SIZE * 20, max_eval_samples=128, num_train_epochs=2, eval_steps=10, logging_steps=10)
    elif QUICK:  # a handful of steps on a CPU
        config.update(max_train_samples=16, max_eval_samples=8, max_steps=4, eval_steps=2, logging_steps=1)
    config.update(changes)
    config = {key: value for key, value in config.items() if value is not None}
    path = WORK_DIR / f"{name}.yaml"
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path


cross_entropy_config = write_config("cross_entropy")
print(cross_entropy_config.read_text())

## Train with cross-entropy

`accelerate launch` starts one process per GPU (DDP) and streams their output here. `DecisionTrainer` scales the loss
and gathers the evaluation across processes; the first process tokenizes the dataset once and the others reuse it.

In [ ]:
def run_environment():
    """The environment of a run: this one's, without anything left by another distributed job."""
    distributed = ("RANK", "LOCAL_RANK", "WORLD_SIZE", "LOCAL_WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT")
    environment = {k: v for k, v in os.environ.items() if not k.startswith("ACCELERATE_") and k not in distributed}
    environment["TOKENIZERS_PARALLELISM"] = "false"
    environment["PYTHONWARNINGS"] = "ignore"
    if FLA_PRESET_DIR is not None:
        environment["FLA_CACHE_MODE"] = "fuzzy"
        environment["FLA_CONFIG_DIR"] = str(Path(FLA_PRESET_DIR).expanduser().resolve())
    return environment


def stream(command):
    """Run a command in a fresh process, print its output as it comes, and stop the notebook if it fails."""
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=WORK_DIR, env=run_environment(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"The command stopped with exit code {process.returncode}; its last lines are above")


def launch(config_path):
    command = [sys.executable, "-m", "accelerate.commands.launch", "--num_processes", PROCESSES, "--num_machines", 1]
    command += ["--mixed_precision", "bf16" if ON_GPU else "no", "--dynamo_backend", "no"]
    if PROCESSES > 1:
        command += ["--multi_gpu", "--main_process_port", 0]
    if not ON_GPU:
        command += ["--cpu"]
    stream([*command, TRAIN_SCRIPT, "--config", config_path])


launch(cross_entropy_config)

## Continue with an RLCD phase

RLCD comes after cross-entropy: this run starts from the model just trained, with a fresh optimizer and lower
learning rates, and anneals the noise per epoch. Its loss keeps the cross-entropy term and adds `RLCD_WEIGHT` times
the RLCD term. A checkpoint keeps its own head, so `head_type` is left out.

In [ ]:
if RLCD_PHASE:
    rlcd_config = write_config(
        "rlcd",
        model_name_or_path=str(WORK_DIR / "cross_entropy"),
        model_revision=None,
        head_type=None,
        num_train_epochs=RLCD_EPOCHS,
        learning_rate=LEARNING_RATE / 5,
        head_learning_rate=HEAD_LEARNING_RATE / 5,
        rlcd_weight=RLCD_WEIGHT,
        rlcd_sigmas=RLCD_SIGMAS,
    )
    launch(rlcd_config)

## Metrics

The final evaluation of each run, on the same held-out records. `selection_score` is accuracy minus a quarter of the
KL divergence and a quarter of the Brier score; it is what picks the checkpoint that is kept.

In [ ]:
METRICS = ["accuracy", "soft_nll", "kl", "brier", "ece", "mean_confidence", "selection_score"]
runs = ["cross_entropy", *(["rlcd"] if RLCD_PHASE else [])]
results = {name: json.loads((WORK_DIR / name / "eval_results.json").read_text()) for name in runs}

print(f"{'':16}" + "".join(f"{name:>16}" for name in METRICS))
for name in runs:
    print(f"{name:16}" + "".join(f"{results[name]['eval_' + metric]:16.4f}" for metric in METRICS))
if QUICK:
    print("\nQUICK run: these numbers only show that the stages work.")

## Ask the trained model

The saved folder is a complete model: the decision pipeline loads it as it is. A request has a `state` and named
`questions` of type `noul` (yes or no), `choice` or `score`.

In [ ]:
import torch
from datasets import load_dataset
from transformers import pipeline

FINAL = WORK_DIR / ("rlcd" if RLCD_PHASE else "cross_entropy")
on_cuda = torch.cuda.is_available()
decide = pipeline("decision", model=str(FINAL), dtype=torch.float32, device=0 if on_cuda else -1)


def ask(request, decide=decide):
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=on_cuda):
        return decide(request)


def as_value(field):
    return json.loads(field) if isinstance(field, str) else field


# A record of the dataset, with what its labels say
row = load_dataset(DATASET, DATASET_CONFIG, revision=DATASET_REVISION, split=f"{EVAL_SPLIT or 'train'}[:1]")[0]
request = {"state": as_value(row["state"]), "questions": as_value(row["questions"])}
print(json.dumps(ask(request), indent=2, default=str)[:3000])
print("gold:", json.dumps(as_value(row["gold"]), default=str)[:1000])

In [ ]:
# Any request of your own
ask(
    {
        "state": "I was charged twice for my order and want a refund.",
        "questions": {
            "refund": {"type": "noul", "instructions": "Is a refund requested?"},
            "topic": {"type": "choice", "instructions": "What is it about?", "criteria": {"billing": None, "shipping": None}},
        },
    }
)

## Optional: convert an older checkpoint

Set `EXISTING_CHECKPOINT` to a folder in the older layout: a backbone with `joint_head.safetensors` and
`joint_head_config.json` (the joint head, or the cosine head when its `head_type` is `cosine_span_v1`). The
converter writes a native folder and never runs code from the source. A trained head on an encoder backbone also
needs `--global_pooling` and `--prompt_style`, because the folder does not record how it was trained.

In [ ]:
if EXISTING_CHECKPOINT is not None:
    converted = WORK_DIR / "converted"
    stream(
        [sys.executable, "-m", "transformers.models.clef.convert_clef_weights_to_hf", "--source", EXISTING_CHECKPOINT, "--output", converted]
    )
    older = pipeline("decision", model=str(converted), dtype=torch.float32, device=0 if on_cuda else -1)
    print(json.dumps(ask(request, older), indent=2, default=str)[:3000])

## Optional: time training steps

Runs the same configuration for a fixed number of steps in one process and reports records per second and the step
time, with the first steps left out.

In [ ]:
if RUN_BENCHMARK:
    steps, excluded = (30, 5) if (QUICK and ON_GPU) else (3, 1) if QUICK else (340, 20)
    stream(
        [sys.executable, BENCHMARK_SCRIPT, "--config", cross_entropy_config, "--steps", steps, "--warmup_steps_excluded", excluded, "--output", WORK_DIR / "benchmark.json"]
    )
    timing = json.loads((WORK_DIR / "benchmark.json").read_text())
    for key in ("window_records_per_second", "steady_step_p50", "steady_step_p95", "peak_allocated_gib", "eval_seconds"):
        print(f"{key:28} {timing.get(key)}")
    print("backbone attention:", timing["recipe"]["attn_implementation"], "| head:", timing["recipe"]["head_type"])

## Notes

**Several GPUs.** `NUM_GPUS = None` uses every visible GPU; the global batch is split evenly, so the optimizer sees
the same number of records per step whatever the count.

| GPUs | Records per GPU | Global batch |
|---|---:|---:|
| 1 | 32 | 32 |
| 2 | 16 | 32 |
| 4 | 8 | 32 |
| 8 | 4 | 32 |

**Kernel tuning.** One of KLM's FLA kernels is tuned again for every new token count of a batch, which costs about
a third of each step on batches it has not seen. `FLA_PRESET_DIR` points FLA at a folder of saved presets (one JSON
file per kernel, as FLA and your `auto_fla_cache.py` write them) and sets `FLA_CACHE_MODE=fuzzy`, so a tuned
configuration is reused for any token count.

**The judge.** `JEV_MONITOR` sends a digest of each evaluation's metrics (never training data) to a System One model
and records its verdicts in `<run>/jev_monitor/ledger.jsonl`. Here it only advises. The key is read from
`TYPESAFE_API_KEY` or `OPENROUTER_API_KEY`.

**In a terminal.** Everything above is plain commands on the files in the run folder (`clef_decision_run/`):

```bash
accelerate launch --multi_gpu --num_processes 8 --mixed_precision bf16 --dynamo_backend no \
    trl/examples/decisions/clef_typed_decisions.py --config cross_entropy.yaml
```

For one GPU leave out `--multi_gpu` and use `--num_processes 1`.